# Test synchronisation Docs → Grist
Notebook de test pour valider chaque étape avant la synchro complète.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, '../src')

from dotenv import load_dotenv
load_dotenv('../.env')

True

## 1. Extraction des chapitres depuis Docs

Trois étapes en une cellule :
1. **Connexion** : initialise le client Docs avec les cookies de session (`.env`)
2. **Récupération du tree** : appelle l'API Docs pour obtenir l'arborescence du document racine
3. **Aplatissement** : parcourt le tree récursivement et produit une liste de records Grist

> Prérequis : `DOCS_SESSION_ID` et `DOCS_CSRF_TOKEN` à jour dans `.env` (F12 → Application → Cookies → `docs.numerique.gouv.fr`)


In [3]:
import os
import docs_client
from importlib import reload

# --- Phase 1 : connexion -------------------------------------------------
# Recharge le module pour prendre en compte les dernières modifs de docs_client.py
reload(docs_client)
from docs_client import DocsClient

docs = DocsClient(
base_url='https://docs.numerique.gouv.fr',
session_id=os.environ.get('DOCS_SESSION_ID'),
csrf_token=os.environ.get('DOCS_CSRF_TOKEN'),
github_token=os.environ.get('GITHUB_TOKEN'), # ← stockage images sur GitHub
github_repo=os.environ.get('GITHUB_REPO', 'nantodevison/docs-suite-numerique'),
github_images_folder=os.environ.get('GITHUB_IMAGES_FOLDER', 'images'),
)
print('DocsClient initialisé.')
if os.environ.get('GITHUB_TOKEN'):
    print(' → Images internes : stockage GitHub activé')
else:
    print(' ⚠️ GITHUB_TOKEN absent → images embarquées en base64')

DocsClient initialisé.
 → Images internes : stockage GitHub activé


## 2. Récupération du tree
Remplace l'URL ci-dessous par celle de ton document racine.

In [4]:
ROOT_URL = 'https://docs.numerique.gouv.fr/docs/21bb0cdd-638f-4a50-8f3a-03616716771e/'

doc_id = DocsClient.extract_doc_id(ROOT_URL)
tree = docs.get_tree(doc_id)

print(f"Racine : {tree['title']}")
print(f"Enfants directs : {len(tree.get('children', []))}")

Racine : Cartographie Stratégique du Bruit - Guide Méthodologique
Enfants directs : 13


## 3. Aplatissement du tree en records Grist

In [5]:
from importlib import reload
import docs_client
reload(docs_client)
from docs_client import DocsClient

# ⚠️ Patch de l'instance existante : après reload(), docs.__class__ pointe encore
# vers l'ancienne classe. On le force à utiliser la nouvelle version rechargée.
docs.__class__ = DocsClient

# ── Choix du mode de récupération du contenu ─────────────────────────────────
# 'json'     → JSON BlockNote → markdown (défaut, préserve callouts et blocs custom)
# 'markdown' → markdown brut (peut échouer sur certains blocs BlockNote custom)
# 'html'     → HTML converti en texte → plus fiable pour les émojis du corps
# 'auto'     → essaie markdown, bascule HTML si le Y-Provider plante
CONTENT_FORMAT = 'json'

records = docs.flatten_tree(tree, 'https://docs.numerique.gouv.fr', is_root=True, content_format=CONTENT_FORMAT)

print(f"\n{len(records)} chapitre(s) extrait(s).")


[racine] Cartographie Stratégique du Bruit - Guide Méthodologique
  [1] 🌟 Guide
    [1.1] 📜 ​Rapportage
    ✓ 1 émoji(s) dans le contenu [json→md]
    [1.2] 🛣️ GITT
    ✓ 1 émoji(s) dans le contenu [json→md]
    [1.3] 🗺️ CBS
    [1.4] 🔠 Glossaire
    [1.5] 📚 ​Bibliographie associée
    [1.6] ℹ️ ​À propos du guide
  [2] 🛠️ Outils
    [2.1] 🔊 NoiseModelling
    [2.2] 💽 Bases de données
    [2.3] 🖥️ Serveur de calcul
  [3] 🏘️ ​🔴 🔵 Bâtiments
    ✓ 2 émoji(s) dans le contenu [json→md]
    [3.1] 🙎‍♂️ ​🔵 Affectation des populations
    ✓ 1 émoji(s) dans le contenu [json→md]
    [3.2] 🧑‍🔧 Implémentation - Bâti
    ✓ 2 émoji(s) dans le contenu [json→md]
  [4] 📍 Récepteurs
    [4.1] 👩‍👦 ​🔵 Décompte de population
    ✓ 1 émoji(s) dans le contenu [json→md]
    [4.2] 🎨 Visuel - isosurfaces
    ✓ 1 émoji(s) dans le contenu [json→md]
    [4.3] 🧑‍🔧 ​🔵 Implémentation - Récepteurs
  [5] 🚙 Routier
    [5.1] 🌐 ​🔴 🔵 Géométrie
    ✓ 3 émoji(s) dans le contenu [json→md]
    [5.2] 📊 ​🔴 Débits
    [5.3] ⏩ ​Vit

### 4.1. Annexe : test d'un record
🧪 Cellule pour tester un record précis

In [6]:

# ── Envoi individuel d'un record (debug — cellule autonome) ──────────────────
from grist_client import GristClient
from docs_client import DocsClient

INDEX = 3  # numéro du record à tester (base 1)

_GRIST_COLUMNS = {'titre', 'niveau', 'ordre', 'numero', 'url', 'contenu'}

_grist = GristClient()
_record = records[INDEX - 1]

_contenu = _record["fields"].get("contenu", "")
if isinstance(_contenu, str):
    _contenu = DocsClient.sanitize_for_waf(_contenu)
_record_clean = {**_record, "fields": {**_record["fields"], "contenu": _contenu}}

print(f"Titre          : {_record_clean['fields'].get('titre')!r}")
print(f"Taille contenu : {len(_contenu)} car.")
print()

ok, ko = _grist.send_records('Chapitres', [_record_clean], columns=_GRIST_COLUMNS)


Titre          : 'GITT'
Taille contenu : 3803 car.

  [00] GITT                                     → OK
  [  1/1] ✓  GITT  (3803 car.)

── Résumé ──────────────────────────────────────────
  Succès : 1
  Échecs : 0


In [5]:
import pandas as pd

df = pd.DataFrame([r['fields'] for r in records])

cols = ['numero', 'niveau', 'emoji', 'titre_propre','titre', 'url', 'contenu_format', 'contenu']
available = [c for c in cols if c in df.columns]
df[available].head(20)


,numero,niveau,emoji,titre_propre,titre,url,contenu_format,contenu
0,1,1,🌟,Guide,🌟 Guide,https://docs.numerique.gouv.fr/docs/242c7d7f-b...,json→md,Ce guide méthodologique est construit autour d...
1,1.1,2,📜,​Rapportage,📜 ​Rapportage,https://docs.numerique.gouv.fr/docs/d3521e51-f...,json→md,## Indicateurs\n\nLa liste des indicateurs att...
2,1.2,2,🛣️,GITT,🛣️ GITT,https://docs.numerique.gouv.fr/docs/e85bb5f4-5...,json→md,"À chaque nouvelle échéance, l'état est en char..."
3,1.3,2,🗺️,CBS,🗺️ CBS,https://docs.numerique.gouv.fr/docs/3c2d989c-4...,json→md,"Au regard de la réglementation, il existe 4 ty..."
4,1.4,2,🔠,Glossaire,🔠 Glossaire,https://docs.numerique.gouv.fr/docs/6b70ef38-7...,json→md,* **CBS** : Carte de Bruit Stratégique\n\n* **...
5,1.5,2,📚,​Bibliographie associée,📚 ​Bibliographie associée,https://docs.numerique.gouv.fr/docs/e8d2b52d-d...,json→md,"## Règlementation\n\nEn Europe, les questions ..."
6,1.6,2,,ℹ️ ​À propos du guide,ℹ️ ​À propos du guide,https://docs.numerique.gouv.fr/docs/bae48920-3...,json→md,## Auteurs : \n\n* Gwendall Petit ([UMRAE](htt...
7,2,1,🛠️,Outils,🛠️ Outils,https://docs.numerique.gouv.fr/docs/1755bc0c-5...,json→md,Dans cette section nous listons les outils gra...
8,2.1,2,🔊,NoiseModelling,🔊 NoiseModelling,https://docs.numerique.gouv.fr/docs/9ee6bf02-1...,json→md,![](https://noisemodelling.readthedocs.io/en/l...
9,2.2,2,💽,Bases de données,💽 Bases de données,https://docs.numerique.gouv.fr/docs/17dcf47e-1...,json→md,## PostGreSQL / PostGIS\n\nL'ensemble des donn...


In [12]:
# ── Diagnostic émojis ─────────────────────────────────────────────────────────
print("Colonnes du df :", list(df.columns))
print()
print("Clés du 1er record :", list(records[0]['fields'].keys()) if records else "records vide")
print()
# Cherche les records qui ont un émoji non vide
with_emoji = [(r['fields'].get('numero'), r['fields'].get('titre'), r['fields'].get('emoji'))
              for r in records if r['fields'].get('emoji')]
print(f"Records avec émoji : {len(with_emoji)}")
for num, titre, em in with_emoji[:10]:
    print(f"  {num} | {em!r} | {titre!r}")

if not with_emoji:
    # Inspecte le titre brut du 1er record pour voir si l'emoji y est
    t0 = records[0]['fields'].get('titre') if records else 'N/A'
    print("\nTitre brut du record 0 :", repr(t0))


Colonnes du df : ['titre', 'emoji', 'titre_propre', 'niveau', 'ordre', 'numero', 'url', 'contenu', 'contenu_format']

Clés du 1er record : ['titre', 'emoji', 'titre_propre', 'niveau', 'ordre', 'numero', 'url', 'contenu', 'contenu_format']

Records avec émoji : 44
  1 | '🌟' | '🌟 Guide'
  1.1 | '📜' | '📜 \u200bRapportage'
  1.2 | '🛣️' | '🛣️ GITT'
  1.3 | '🗺️' | '🗺️ CBS'
  1.4 | '🔠' | '🔠 Glossaire'
  1.5 | '📚' | '📚 \u200bBibliographie associée'
  2 | '🛠️' | '🛠️ Outils'
  2.1 | '🔊' | '🔊 NoiseModelling'
  2.2 | '💽' | '💽 Bases de données'
  2.3 | '🖥️' | '🖥️ Serveur de calcul'


In [6]:
from grist_client import GristClient

grist = GristClient()
_GRIST_COLUMNS = {'titre', 'niveau', 'ordre', 'numero', 'url', 'contenu'}
ok, ko = grist.send_records('Chapitres', records, columns=_GRIST_COLUMNS)
print(f"Enregistrements ajoutés : {ok}, échecs : {ko}")

  [00] 🌟 Guide                                  → OK
  [  1/50] ✓  🌟 Guide  (605 car.)
  [00] 📜 ​Rapportage                            → OK
  [  2/50] ✓  📜 ​Rapportage  (3575 car.)
  [00] 🛣️ GITT                                  → OK
  [  3/50] ✓  🛣️ GITT  (5708 car.)
  [00] 🗺️ CBS                                   → OK
  [  4/50] ✓  🗺️ CBS  (811 car.)
  [00] 🔠 Glossaire                              → OK
  [  5/50] ✓  🔠 Glossaire  (1179 car.)
  [00] 📚 ​Bibliographie associée                → OK
  [  6/50] ✓  📚 ​Bibliographie associée  (8992 car.)
  [00] ℹ️ ​À propos du guide                    → OK
  [  7/50] ✓  ℹ️ ​À propos du guide  (517 car.)
  [00] 🛠️ Outils                                → OK
  [  8/50] ✓  🛠️ Outils  (245 car.)
  [00] 🔊 NoiseModelling                         → OK
  [  9/50] ✓  🔊 NoiseModelling  (771 car.)
  [00] 💽 Bases de données                       → OK
  [ 10/50] ✓  💽 Bases de données  (936 car.)
  [00] 🖥️ Serveur de calcul                     → OK
  [ 11/50]

In [12]:
records

[{'fields': {'titre': 'Guide',
   'emoji': '🌟',
   'titre_propre': 'Guide',
   'niveau': 1,
   'ordre': 1,
   'numero': '1',
   'url': 'https://docs.numerique.gouv.fr/docs/242c7d7f-b715-48bc-aa06-677a8dfc9ad0/',
   'contenu': "Ce guide méthodologique est construit autour de la structure suivante :\n\nQuelques éléments de contexte et de vocabulaire :\n\n*\n\n*\n\n*\n\n*\n\n*\n\n*\n\nLes utilisés :\n\n* Modélisation du bruit :\n\n* Stockage et gestion des données :\n\n* Exécution des calculs :\n\nLes données d'entrée :\n\n*\n\n*\n\n*\n\n*\n\n*\n\n*\n\n*\n\n*\n\n*\n\nLes\n",
   'contenu_format': 'markdown'}},
 {'fields': {'titre': 'Rapportage',
   'emoji': '📜',
   'titre_propre': '\u200bRapportage',
   'niveau': 2,
   'ordre': 0,
   'numero': '1.1',
   'url': 'https://docs.numerique.gouv.fr/docs/d3521e51-f6eb-49b4-8337-d19c643a71cd/',
   'contenu': '## Indicateurs\n\nLa liste des indicateurs attendus est détaillée ici .\n\n## Échelle de rapportage\n\n### Rapportage des populations et des 